# Reading a `LibraryReport`: real deposits, no download

> ℹ️ **Illustrative tour of real `detect` outputs — _not_ benchmark evidence.**
> These are cached `LibraryReport` JSONs that `selexprep detect` 0.4.7 produced on
> real public deposits (`examples/regenerate_reports.sh` regenerates them), bundled
> here so you can see realistic behavior with **no network or HPC**. They show the
> report *format and the range of outcomes*, not a performance claim —
> primer-recovery *performance* on paper-documented deposits is established in
> **the Tier 1 scorecard** (`benchmarks/`).

A `LibraryReport` answers three questions: did `selexprep` find the library's
primers, how confident is it, and what should you do next? The four real
deposits below span the outcome range.

## The fields that matter

- **`status`** — `HIGH` / `MEDIUM` / `LOW` / `UNABLE_TO_INFER`: overall confidence.
- **`extraction_mode`** — *biology*: `BOTH_PRIMERS_SINGLE_READ`, `FIVE_PRIME_ONLY`,
  `THREE_PRIME_ONLY`, `PAIRED_END_SPLIT_PRIMERS`, `UNABLE_TO_EXTRACT`.
- **`required_action`** — *workflow*: `NONE`, `MANUAL_PRIMERS_REQUIRED`, `READ_MERGING_RECOMMENDED`.
- **`full_insert_recovered`** — were both flanks found in a single read?
- **`known_adapter_hits`** — sequencing adapters seen (recorded, never silently called as primers).
- **`match_rate_*` / `position_consistency_*`** — how cleanly each constant sits at the read end.

Design point: `extraction_mode` (biology), `required_action` (workflow), and
`read_source` (file layout) are **separate axes** — read them together.

In [1]:
import json, pathlib

def show(acc):
    r = json.load(open(pathlib.Path("data") / f"{acc}.library_report.json"))
    keys = ["status", "extraction_mode", "read_source", "required_action",
            "full_insert_recovered", "primer_5p", "primer_3p",
            "match_rate_5p", "match_rate_3p", "n_length_mode", "known_adapter_hits"]
    print(f"# {acc}")
    for k in keys:
        print(f"  {k:22} {r[k]}")
    if r.get("failure_reason"):
        print(f"  {'failure_reason':22} {r['failure_reason']}")
    return r

## 1. A clean recovery — `HIGH`  (PRJNA615076, DNA whole-cell SELEX)

In [2]:
_ = show("PRJNA615076")

# PRJNA615076
  status                 HIGH
  extraction_mode        BOTH_PRIMERS_SINGLE_READ
  read_source            R1_AND_R2
  required_action        NONE
  full_insert_recovered  True
  primer_5p              TAGGGAAGAGAAGGACATATGAT
  primer_3p              TTGACTAGTACATGACCACTTGA
  match_rate_5p          0.9304520708300043
  match_rate_3p          0.9293465133131422
  n_length_mode          40
  known_adapter_hits     {'NEXTERA': 1854, 'TRUSEQ_R1': 0}


Both primers recovered, `extraction_mode = BOTH_PRIMERS_SINGLE_READ`,
`full_insert_recovered = True`, `required_action = NONE` → `extract` proceeds
with no manual input. (These happen to match the paper's primers — but checking
that is the Tier 1 scorecard's job, not this notebook's.)

## 2. Supported by the reads, not identical to the paper — `HIGH`  (PRJEB62495, DNA whole-cell SELEX)

In [3]:
_ = show("PRJEB62495")

# PRJEB62495
  status                 HIGH
  extraction_mode        BOTH_PRIMERS_SINGLE_READ
  read_source            R1
  required_action        NONE
  full_insert_recovered  True
  primer_5p              GGCTTCTGGACTACCTATGC
  primer_3p              CGTGGTTACAGTCAGAGGA
  match_rate_5p          0.9730360686354617
  match_rate_3p          0.9103944014485004
  n_length_mode          40
  known_adapter_hits     {'NEXTERA': 0, 'TRUSEQ_R1': 410}


Both constants are called and the random region is 40 nt, as published. The
3′ call, `CGTGGTTACAGTCAGAGGA`, is the published 3′ constant without its last
base (`…GAGGAG`): that base differs between the deposit's runs, and `detect`
keeps only the part of a flank that every round shares. `status = HIGH` says the
reads support the call well; it does not say the call equals the publication.
When the constants matter, compare them with the paper.

## 3. A one-round, paired-end deposit — `MEDIUM`  (PRJNA1395820, DNA, Co²⁺)

In [4]:
_ = show("PRJNA1395820")

# PRJNA1395820
  status                 MEDIUM
  extraction_mode        PAIRED_END_SPLIT_PRIMERS
  read_source            R1_AND_R2
  required_action        READ_MERGING_RECOMMENDED
  full_insert_recovered  False
  primer_5p              TACACTGCACTGCGTTAGAG
  primer_3p              ATGGACGTGTCACAGTACTG
  match_rate_5p          0.980929975427288
  match_rate_3p          0.9939938579157852
  n_length_mode          None
  known_adapter_hits     {'NEXTERA': 7, 'TRUSEQ_R1': 5}


Only one round is deposited, so cross-round persistence is unavailable and
`status` is capped at `MEDIUM`. The 5′ constant sits on R1 and the 3′ constant on
R2 (`read_source = R1_AND_R2`), with no overlap between the mates:
`extraction_mode = PAIRED_END_SPLIT_PRIMERS`, `full_insert_recovered = False`, no
random-region length, and `required_action = READ_MERGING_RECOMMENDED`.
`extract` trims each side separately; joining them into full inserts needs a read
merger, which `selexprep` does not provide, so `count` and `qc` are skipped for
these rounds rather than counting half-inserts.

## 4. A safe failure — `UNABLE_TO_INFER`  (PRJEB70964, pre-trimmed reads)

In [5]:
_ = show("PRJEB70964")

# PRJEB70964
  status                 UNABLE_TO_INFER
  extraction_mode        UNABLE_TO_EXTRACT
  read_source            R1
  required_action        MANUAL_PRIMERS_REQUIRED
  full_insert_recovered  False
  primer_5p              None
  primer_3p              None
  match_rate_5p          0.0
  match_rate_3p          0.0
  n_length_mode          35
  known_adapter_hits     {'NEXTERA': 8, 'TRUSEQ_R1': 66}
  failure_reason         No candidate 5' or 3' flank found in the earliest round


The depositor uploaded only the 35-nt randomised region, so these reads carry
no constant for `detect` to find: both match rates are 0 and the random-region
length it reports is simply the read length. Rather than invent a flank it
returns `primer_5p/3p = null`, `status = UNABLE_TO_INFER` (`failure_reason`: no
candidate flank in the earliest round) and
`required_action = MANUAL_PRIMERS_REQUIRED`, and **downstream `extract`
refuses** unless you pass `--override-primer-5p/-3p`. For a deposit like this
one there is nothing to trim: count the reads directly with
`selexprep count --from-pretrimmed-fastq`.

## Takeaway
`HIGH → extract proceeds` (still check the constants against the paper) ·
`MEDIUM → proceeds, mind the caveats` (here: one round, split mates) ·
`UNABLE_TO_INFER → refuses, asks for manual primers`. The safe-failure path —
refusing rather than fabricating a primer — is the property known-primer
pipelines can't offer. For recovery *performance* on paper-documented deposits,
see the Tier 1 scorecard.